In [0]:
%run ./world_export_config


In [0]:
raw_path = f"abfss://{CONTAINER}@{storage_account}.dfs.core.windows.net/rawData/world_exports_raw.csv"
raw_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(raw_path)

raw_df.printSchema()

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# Bronze layer: load as-is, just add ingestion metadata
# Never modify source data in Bronze — this is the law of Medallion Architecture
bronze_df = raw_df \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.lit("world_exports_raw.csv")) \
    .withColumn("layer", F.lit("bronze"))

bronze_path = f"abfss://{CONTAINER}@{storage_account}.dfs.core.windows.net/bronze/world_exports"

# Save as Delta table to ADLS Gen2
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

print(f"Bronze layer saved: {bronze_df.count()} records")
display(bronze_df.limit(10))